<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/04a_ragas_aspect_critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4a: Custom LLM-as-a-Judge: RAGAS AspectCritic

**Goal:** Build a custom RAGAS AspectCritic evaluator using Claude as the
judge model. AspectCritic evaluates responses against named, domain-specific
aspects rather than generic RAG quality metrics. Here the aspects are drawn
directly from the EU AI Act and NIST AI RMF obligations established in the
Phase 3b G-Eval rubrics.

**Tools:** RAGAS AspectCritic, Claude (claude-sonnet-4-6) as judge

**Aspects evaluated:**
- correctness: factual accuracy relative to retrieved regulatory documents
- regulatory_grounding: claims traceable to specific regulatory articles
- oversight_representation: Article 14 human oversight accurately represented
- bias_representation: Article 10 data governance accurately represented
- harm_potential: does the response risk misleading a deployer about compliance

**Design addition (Federico Blanco Sanchez-Llanos):** The two-queue split
(quality failures route to retrieval/generation layer, compliance failures
route to governance layer) must survive independent of whoever made the call.
This notebook exports each AspectCritic verdict as a signed artifact bound
to a hash of the specific inputs so the routing decision is independently
auditable.

**SIMULATED_OUTPUT flag:** Set to True throughout.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm prior phases

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase3b_path = DRIVE_PATH + "phase03b_governance_metrics_results.json"
if os.path.exists(phase3b_path):
    with open(phase3b_path) as f:
        phase3b = json.load(f)
    print("Phase 3b results confirmed.")
    print(f"  Outcome accuracy: {phase3b['overall']['outcome_accuracy']}")
    print(f"  Adversarial detection: "
          f"{phase3b['overall']['adversarial_detection_rate']}")
    print(f"  Artifact limitation: "
          f"{phase3b['governance_evaluation']['artifact_limitation'][:80]}...")
else:
    print("WARNING: Phase 3b results not found.")
    print(f"Expected: {phase3b_path}")
    print("Run 03b_deepeval_governance_metrics.ipynb first.")

Mounted at /content/drive
Phase 3b results confirmed.
  Outcome accuracy: 6/7
  Adversarial detection: 3/3
  Artifact limitation: All compliance verdicts are bound to SHA-256 input hashes. Hashes prove non-alte...
